In [ ]:
library(rtracklayer)
library(ggplot2)
library(Seurat)

getwd()
dir.create("figures_10xMouse_PBMC")
dir.create("data")
colorDict = c("2CLC"="#88535A",
              "Zscan4+"="#EF8264",
              "Pluri"="#F2CC8F")

colorAge = c("old"="#A58065", #A67C63, #DB9D74
             "young"="#7FCFF2") #049DD9

colorTools = c("SoloTE"="#A4DD9B", #6fc69d",
               "Stellarscope"="#4f5d93",
               "STARsolo"="#f0df93")

dataset_id <- "10xMouse_RA"

In [ ]:
# theme_paper <- function(base_size = 17, base_family = "") {
#   theme_minimal(base_size = base_size, base_family = base_family) +
#     theme(
#       plot.title = element_text(face = "bold", size = base_size + 2, hjust = 0.5),
#       axis.title = element_text(size = base_size),
#       axis.text  = element_text(size = base_size * 0.9),
#       legend.title = element_text(size = base_size),
#       legend.text  = element_text(size = base_size * 0.9),
#       panel.grid.major = element_line(linewidth = 0.4),
#       panel.grid.minor = element_blank(),
#       plot.margin = margin(10, 10, 10, 10)
#     )
# }

# theme_set(theme_paper())

In [ ]:
#save.image("workspaces/afterCorrelations_wSTAR.RData")
load("workspaces/afterCorrelations_wSTAR_thr5percCells_10xMouse_PBMC.RData")
rm(mat_stellarscope)
rm(mat_soloTE)
rm(mat_STARsolo)
gc()


# Gene Intersections

In [ ]:
library(rtracklayer)
# import the annotation used for snakemake
gene_annotation <- rtracklayer::import("/mnt/TEdataStorage_2T/snakemake_data/references/mm10/gencode.vM10.annotation.gtf.gz")
names(mcols(gene_annotation))

In [ ]:
gene_annotation_df <- as.data.frame(gene_annotation)

# Extract only exon entries
exons <- gene_annotation[gene_annotation$type == "exon"]
names(exons) <- exons$gene_name

In [ ]:
# keep only TEs that are in both SoloTE and Stellarscope
annotation_common <- na.omit(conversionTable)

cat("Classes not in SoloTE annotation: ", 
    paste(setdiff(unique(conversionTable$class), unique(annotation_common$class)), collapse = ", "), 
    "\n")

In [ ]:
objTElist <- list()

objTElist[["SoloTE"]] <- objTE_soloTE
objTElist[["Stellarscope"]] <- objTE_stellarscope
objTElist[["STARsolo"]] <- objTE_STARsolo

objTElist

In [ ]:
# Compute overlaps to exons and save lists of loci
featuresList <- list()

bedList <- list()
grList <- list()
exonOverlapList <- list()
toolOverlappingGrList <- list()
toolOverlappingList <- list()

for(tool in names(objTElist)){
    if(tool == "SoloTE"){
        bedList[[tool]] <- conversionTable[conversionTable$soloteID %in% Features(objTElist[[tool]]), 
                            c("chr","start","end","strand","stellarscopeID")]
        print(nrow(bedList[[tool]]))
    }else{
        bedList[[tool]] <- conversionTable[conversionTable$stellarscopeID %in% Features(objTElist[[tool]]), 
                            c("chr","start","end","strand","stellarscopeID")]
        print(nrow(bedList[[tool]]))
    }

    bedList[[tool]] <- bedList[[tool]][!duplicated(bedList[[tool]]$stellarscopeID),]
    bedList[[tool]] <- bedList[[tool]][!is.na(bedList[[tool]]$stellarscopeID),]
    featuresList[[tool]] <- bedList[[tool]]$stellarscopeID
    print(nrow(bedList[[tool]]))
    rownames(bedList[[tool]]) <- bedList[[tool]]$stellarscopeID
    # turn into GR object
    grList[[tool]] <- makeGRangesFromDataFrame(bedList[[tool]])
    exonOverlapList[[tool]] <- findOverlaps(grList[[tool]], exons, minoverlap = 10)

    toolOverlappingGrList[[tool]] <- grList[[tool]][unique( exonOverlapList[[tool]]@from)] 
    toolOverlappingList[[tool]] <- rownames(as.data.frame(toolOverlappingGrList[[tool]]))
    #print(as.data.frame(toolOverlappingList[[tool]]))
}


In [ ]:
colorTools

In [ ]:
library(UpSetR)
options(repr.plot.width=9, repr.plot.height=6)

ph <- UpSetR::upset(fromList(toolOverlappingList), 
    sets=names(toolOverlappingList), sets.bar.color=colorTools,
    keep.order = TRUE,
    text.scale = c(2, 2, 2, 1.75, 2.75, 2.5), 
    order.by=c("freq"),
    point.size=4, mb.ratio = c(0.63, 0.37),
    set_size.show=F, set_size.scale_max= max(lengths(toolOverlappingList))+2000, 
    sets.x.label="N. loci intersecting \n gene exons",
    mainbar.y.label="Intersection size")

pdf(paste0("figures_", dataset_id, "/NlociIntersectingGeneExons_upset.pdf"), width=9, height=6)
ph
dev.off()
ph

In [ ]:
df <- NULL

for(tool in names(objTElist)){
    
    loci <- bedList[[tool]]$stellarscopeID
    lociOverlappingGenes <- toolOverlappingList[[tool]]

    df <- rbind(df, c(tool, length(loci), length(lociOverlappingGenes)))

}

df <-  as.data.frame(df)
df